# 01 · Bronze layer: India Automotive Intelligence Platform

**What this notebook does:** reads all the raw files you downloaded (`IAIP\bronze\...`), turns each wide table into one tidy **long** table, checks the structure of every file, and saves everything as **Parquet** in `IAIP\warehouse\bronze\`.

**Bronze rule:** keep the data **exactly as it came** from the source. No renaming of states, no number conversion, no deleting rows. Every row remembers **which file it came from**. Cleaning happens later, in Silver.

| Raw file (wide) | Bronze table (long) |
|---|---|
| one row per state, one column per month | one row per **state × month** |

**How to run:** top to bottom, one cell at a time (Shift + Enter). The whole notebook takes about 2–5 minutes.

## Step 1 · Start Spark and set the folders

In [1]:
import os, sys, re, csv
from pathlib import Path
from datetime import datetime
from functools import reduce

# Windows: tell Spark which Python to use
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, DataFrame, functions as F, types as T

spark = (SparkSession.builder
         .appName("IAIP-01-Bronze")
         .master("local[*]")                              # use all CPU cores of this laptop
         .config("spark.sql.shuffle.partitions", "8")     # small data: 8 partitions is plenty
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

BASE = Path.cwd().parent            # ...\Desktop\IAIP   (this notebook lives in IAIP\notebooks)
RAW  = BASE / "bronze"              # the raw files you downloaded (landing zone)
WH   = BASE / "warehouse"           # our Parquet warehouse: bronze / silver / gold
assert RAW.exists(), f"Raw folder not found at {RAW}. Open this notebook from IAIP\\notebooks."

LOAD_TS = datetime.now().strftime("%Y-%m-%d %H:%M:%S")   # when this load ran (lineage)
print("Spark", spark.version, "| raw:", RAW, "| warehouse:", WH)

c:\Users\Arka\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark 4.2.0 | raw: c:\Users\Arka\Desktop\IAIP\bronze | warehouse: c:\Users\Arka\Desktop\IAIP\warehouse


## Step 2 · Read the description line of each file (metadata)

Every VAHAN file starts with a long description such as *"Fuel (DIESEL), Emission (...), Type (Transport) ... (2020-2022)"*. It records **every filter used**, so we read it to:
- check that the file really is what its name says (Transport vs Non-Transport, which years, how many fuels)
- find the real column headers

There are two layouts:
- **Layout A** (fuel, category, norm files): line 1 = description + column headers
- **Layout B** (maker files): line 1 = description, line 2 = empty, line 3 = column headers

This step reads only the first 3 lines of each file with plain Python, which is fast. Spark reads the actual data in Step 3.

In [2]:
def read_header(path):
    """Return the description text, layout and column headers of one VAHAN file."""
    with open(path, encoding="utf-8-sig", newline="") as f:
        first3 = [row for _, row in zip(range(3), csv.reader(f))]
    text = first3[0][0]
    if len(first3[0]) > 1:                     # Layout A
        layout, value_cols = "A", first3[0][1:]
    else:                                      # Layout B (maker files)
        layout, value_cols = "B", first3[2][1:]
    vtype = "NT" if "Type (Non-Transport)" in text else ("T" if "Type (Transport)" in text else None)
    years = re.findall(r"\((\d{4}(?:-\d{4})?)\)", text)
    fuels = re.match(r"Fuel \((.*?)\), (?:Emission|Category Group)", text)   # "Emission (...)" is left out when that filter is empty
    return {"text": text, "layout": layout, "value_cols": value_cols,
            "vtype_in_header": vtype,
            "years_in_header": years[-1] if years else None,
            "n_fuels_in_header": len(fuels.group(1).split(", ")) if fuels else None}

# quick test on one file
h = read_header(RAW / "vahan_state_month" / "fuel_EV_T_2026.csv")
print(h["layout"], h["vtype_in_header"], h["years_in_header"], h["n_fuels_in_header"], h["value_cols"][:4])

A T 2026 2 ['2026-Jan', '2026-Feb', '2026-Mar', '2026-Apr']


## Step 3 · Read one file with Spark and make it long

1. **Spark reads the CSV.** We give it the exact number of columns (a *schema*), all as text. That's Bronze: no conversion yet.
   `escape='"'` handles maker names that contain commas or quotes (452 of them), and `multiLine=True` handles the 10 maker names that contain a hidden line break.
2. Drop the description, blank and header lines. Spark reads the description line slightly differently from Python (it can keep a leading quote mark, and maker files start with an invisible "byte order mark"), so we find it by its text *"Fuel ("* anywhere in the cell.
3. **Unpivot** (wide → long): each `state × month` cell becomes its own row.
4. Add **lineage** columns: source file, dataset, vehicle type, fuel group, load time.

In [3]:
def read_vahan_file(path, dataset, fuel_group=None, vtype_from_name=None):
    h = read_header(path)
    n_cols = 1 + len(h["value_cols"])
    safe = [f"c{i}" for i in range(n_cols)]                      # safe temporary column names
    schema = T.StructType([T.StructField(c, T.StringType()) for c in safe])

    df = (spark.read
          .option("header", False)
          .option("escape", '"')            # "ABC PVT, LTD" style names
          .option("multiLine", True)        # 10 maker names contain a line break inside quotes
          .option("mode", "PERMISSIVE")
          .schema(schema)
          .csv(path.as_uri()))

    # remove the invisible "byte order mark" (\ufeff) that starts each file
    df = df.withColumn("c0", F.regexp_replace(F.col("c0"), "^\ufeff", ""))

    # drop the description line, the empty line and the header line
    df = df.filter(F.col("c0").isNotNull()
                   & (F.trim(F.col("c0")) != "")
                   & (~F.col("c0").contains("Fuel ("))     # the description line, however Spark split it
                   & (~F.col("c0").isin("Maker", "State")))

    # wide -> long
    long = df.unpivot(ids=["c0"], values=safe[1:],
                      variableColumnName="col_id", valueColumnName="value_raw")

    # put the real column labels back (e.g. c1 -> "2020-Jan")
    label_map = F.create_map(*[F.lit(x) for pair in zip(safe[1:], h["value_cols"]) for x in pair])
    long = (long.withColumn("column_label", F.element_at(label_map, F.col("col_id")))
                .withColumnRenamed("c0", "row_label")
                .drop("col_id")
                .withColumn("is_total", (F.col("row_label") == "Total") | (F.col("column_label") == "Total"))
                .withColumn("dataset", F.lit(dataset))
                .withColumn("vehicle_type", F.lit(vtype_from_name))
                .withColumn("fuel_group", F.lit(fuel_group))
                .withColumn("source_file", F.lit(path.name))
                .withColumn("load_ts", F.lit(LOAD_TS)))
    return long, h

# quick test
test_df, _ = read_vahan_file(RAW / "vahan_state_month" / "fuel_EV_T_2026.csv", "reg_state_month", "EV", "T")
test_df.show(5, truncate=False)
print("rows:", test_df.count(), "(expected 37 rows x 11 columns = 407)")

+------------------------+---------+------------+--------+---------------+------------+----------+------------------+-------------------+
|row_label               |value_raw|column_label|is_total|dataset        |vehicle_type|fuel_group|source_file       |load_ts            |
+------------------------+---------+------------+--------+---------------+------------+----------+------------------+-------------------+
|Andaman & Nicobar Island|1        |2026-Jan    |false   |reg_state_month|T           |EV        |fuel_EV_T_2026.csv|2026-11-13 21:49:24|
|Andaman & Nicobar Island|6        |2026-Feb    |false   |reg_state_month|T           |EV        |fuel_EV_T_2026.csv|2026-11-13 21:49:24|
|Andaman & Nicobar Island|5        |2026-Mar    |false   |reg_state_month|T           |EV        |fuel_EV_T_2026.csv|2026-11-13 21:49:24|
|Andaman & Nicobar Island|2        |2026-Apr    |false   |reg_state_month|T           |EV        |fuel_EV_T_2026.csv|2026-11-13 21:49:24|
|Andaman & Nicobar Island|7       

## Step 4 · Load all four VAHAN datasets (84 files)

| Bronze table | Folder | Rows (labels) | Columns (labels) |
|---|---|---|---|
| `reg_state_month` | vahan_state_month | state | month (`2024-Jan`) |
| `reg_state_category_year` | vahan_category | state | vehicle category |
| `reg_state_norm_year` | vahan_norms | state | emission norm |
| `reg_maker_state_year` | vahan_maker | **maker** | **state** |

The loop also records a **manifest**: one line per file with what we expected and what we found.

In [4]:
SOURCES = {
    "reg_state_month":         ("vahan_state_month", r"^fuel_([A-Z]+)_(T|NT)_(\d{4}(?:_\d{4})?)\.csv$"),
    "reg_state_category_year": ("vahan_category",    r"^vcat_(T|NT)_(\d{4})\.csv$"),
    "reg_state_norm_year":     ("vahan_norms",       r"^norm_(T|NT)_(\d{4})\.csv$"),
    "reg_maker_state_year":    ("vahan_maker",       r"^maker_(T|NT)_(\d{4})\.csv$"),
}
EXPECTED_FUELS = {"ALL": 34, "EV": 2, "PETROL": 2, "DIESEL": 1, "HYBRID": 7, "GAS": 12, "OTHER": 10}

tables, manifest = {}, []
for dataset, (folder, pattern) in SOURCES.items():
    frames = []
    for path in sorted((RAW / folder).glob("*.csv")):
        m = re.match(pattern, path.name)
        if not m:
            print("  skipped (name not recognised):", path.name); continue
        if dataset == "reg_state_month":
            fuel_group, vtype, period = m.group(1), m.group(2), m.group(3).replace("_", "-")
        else:
            fuel_group, vtype, period = "ALL", m.group(1), m.group(2)
        df, h = read_vahan_file(path, dataset, fuel_group, vtype)
        frames.append(df)
        manifest.append({
            "source_file": path.name, "dataset": dataset, "layout": h["layout"],
            "vehicle_type": vtype, "fuel_group": fuel_group, "period_in_name": period,
            "vtype_in_header": h["vtype_in_header"], "years_in_header": h["years_in_header"],
            "n_fuels_in_header": h["n_fuels_in_header"], "n_value_columns": len(h["value_cols"]),
            "last_column": h["value_cols"][-1], "file_bytes": path.stat().st_size,
            "file_modified": datetime.fromtimestamp(path.stat().st_mtime).strftime("%Y-%m-%d %H:%M"),
            "filter_text": h["text"], "load_ts": LOAD_TS})
    tables[dataset] = reduce(DataFrame.unionByName, frames)
    print(f"{dataset:26} {len(frames):3} files")

reg_state_month             42 files
reg_state_category_year     14 files
reg_state_norm_year         14 files
reg_maker_state_year        14 files


## Step 5 · Structure checks (proposal: "Bronze checks their structure")

For every file we check:
- **type_ok**: Transport/Non-Transport in the description matches the file name
- **years_ok**: the years in the description match the file name
- **fuels_ok**: the right number of fuels were ticked (EV 2, GAS 12, ALL 34, …)
- **states_ok**: exactly **36 states**
- **total_ok**: a Total row/column exists (we use it for checks in Silver)

This is the first part of your **E3 (data quality)** results.

In [5]:
man = spark.createDataFrame(manifest)

# count rows per file and states per file, using Spark
stats = []
for dataset, df in tables.items():
    if dataset == "reg_maker_state_year":   # states are the COLUMNS in maker files
        s = df.filter(~F.col("is_total")).groupBy("source_file").agg(
                F.count("*").alias("bronze_rows"),
                F.countDistinct("column_label").alias("n_states"),
                F.countDistinct("row_label").alias("n_makers"))
    else:                                   # states are the ROWS
        s = df.filter(~F.col("is_total")).groupBy("source_file").agg(
                F.count("*").alias("bronze_rows"),
                F.countDistinct("row_label").alias("n_states"),
                F.lit(None).cast("long").alias("n_makers"))
    stats.append(s)
stats = reduce(DataFrame.unionByName, stats)

fuel_expected = F.create_map(*[F.lit(x) for kv in EXPECTED_FUELS.items() for x in kv])
checks = (man.join(stats, "source_file", "left")
    .withColumn("type_ok",  F.col("vtype_in_header") == F.col("vehicle_type"))
    .withColumn("years_ok", F.col("years_in_header") == F.col("period_in_name"))
    .withColumn("fuels_ok", F.col("n_fuels_in_header") == F.when(F.col("dataset") == "reg_state_month",
                                                                 F.element_at(fuel_expected, F.col("fuel_group"))).otherwise(F.lit(34)))
    .withColumn("states_ok", F.col("n_states") == 36)
    .withColumn("total_ok",  F.col("last_column") == "Total")
    .withColumn("all_ok", F.col("type_ok") & F.col("years_ok") & F.col("fuels_ok") & F.col("states_ok") & F.col("total_ok")))

checks.groupBy("dataset").agg(F.count("*").alias("files"),
                              F.sum(F.coalesce(F.col("all_ok"), F.lit(False)).cast("int")).alias("files_passing"),
                              F.sum("bronze_rows").alias("rows")).orderBy("dataset").show(truncate=False)

print("Files failing any check:")
checks.filter(~F.coalesce(F.col("all_ok"), F.lit(False))).select("source_file", "type_ok", "years_ok", "fuels_ok", "states_ok", "total_ok").show(truncate=False)

+-----------------------+-----+-------------+------+
|dataset                |files|files_passing|rows  |
+-----------------------+-----+-------------+------+
|reg_maker_state_year   |14   |14           |532332|
|reg_state_category_year|14   |14           |8568  |
|reg_state_month        |42   |42           |41832 |
|reg_state_norm_year    |14   |14           |13104 |
+-----------------------+-----+-------------+------+

Files failing any check:
+-----------+-------+--------+--------+---------+--------+
|source_file|type_ok|years_ok|fuels_ok|states_ok|total_ok|
+-----------+-------+--------+--------+---------+--------+
+-----------+-------+--------+--------+---------+--------+



## Step 6 · Road-safety files (OpenCity / MoRTH)

These are normal CSVs with a header row, so Spark reads them directly. We again **unpivot** the year columns (`2019 Accidents`, `2020 Accidents` …) into rows and keep the values as text. Footnote rows (like "# includes Ladakh…") are kept too; Silver will deal with them.

In [6]:
def snake(name):
    """'Road accidents (\'000)' -> 'road_accidents_000' (Parquet-safe column names)."""
    return re.sub(r"_+", "_", re.sub(r"[^0-9a-zA-Z]+", "_", name.replace("﻿", ""))).strip("_").lower()

def read_road_state(path, measure):
    df = spark.read.option("header", True).option("escape", '"').csv(path.as_uri())
    df = df.toDF(*[c.replace("﻿", "") for c in df.columns])
    year_cols = [c for c in df.columns if re.match(rf"^\d{{4}} {measure}$", c)]
    long = (df.select(F.col("Sl No").alias("sl_no"), F.col("State").alias("state_raw"), *year_cols)
              .unpivot(ids=["sl_no", "state_raw"], values=year_cols,
                       variableColumnName="year_label", valueColumnName="value_raw")
              .withColumn("year", F.substring("year_label", 1, 4))
              .withColumn("measure", F.lit(measure.lower()))
              .withColumn("is_total", F.col("sl_no") == "Total")
              .withColumn("source_file", F.lit(path.name))
              .withColumn("load_ts", F.lit(LOAD_TS)))
    return long

RS = RAW / "road_safety"
road_state = (read_road_state(RS / "road_accidents_state_2019_2023.csv", "Accidents")
              .unionByName(read_road_state(RS / "road_fatalities_state_2019_2023.csv", "Killed")))

road_nat = spark.read.option("header", True).option("escape", '"').csv((RS / "road_accidents_vehicles_density.csv").as_uri())
road_nat = road_nat.toDF(*[snake(c) for c in road_nat.columns]) \
                   .withColumn("source_file", F.lit("road_accidents_vehicles_density.csv")) \
                   .withColumn("load_ts", F.lit(LOAD_TS))

road_state.filter(F.col("state_raw") == "Kerala").show(truncate=False)
print("road_state rows:", road_state.count(), "| national rows:", road_nat.count())
road_nat.printSchema()

+-----+---------+--------------+---------+----+---------+--------+-----------------------------------+-------------------+
|sl_no|state_raw|year_label    |value_raw|year|measure  |is_total|source_file                        |load_ts            |
+-----+---------+--------------+---------+----+---------+--------+-----------------------------------+-------------------+
|12   |Kerala   |2019 Accidents|41111    |2019|accidents|false   |road_accidents_state_2019_2023.csv |2026-11-13 21:49:24|
|12   |Kerala   |2020 Accidents|27877    |2020|accidents|false   |road_accidents_state_2019_2023.csv |2026-11-13 21:49:24|
|12   |Kerala   |2021 Accidents|33296    |2021|accidents|false   |road_accidents_state_2019_2023.csv |2026-11-13 21:49:24|
|12   |Kerala   |2022 Accidents|43910    |2022|accidents|false   |road_accidents_state_2019_2023.csv |2026-11-13 21:49:24|
|12   |Kerala   |2023 Accidents|48091    |2023|accidents|false   |road_accidents_state_2019_2023.csv |2026-11-13 21:49:24|
|12   |Kerala   

## Step 7 · Save everything as Parquet (the Bronze warehouse)

- `mode("overwrite")`: running this notebook again **replaces** the Bronze tables instead of adding duplicates. Running twice gives the same result, which is what your proposal promises.
- `partitionBy`: Spark stores each vehicle type (and fuel group) in its own sub-folder, so later reads can skip what they don't need.

In [7]:
def save(df, name, partition_by=None):
    out = (WH / "bronze" / name).as_uri()
    writer = df.write.mode("overwrite")
    if partition_by:
        writer = writer.partitionBy(*partition_by)
    writer.parquet(out)
    print(f"saved bronze/{name}")

save(tables["reg_state_month"],         "reg_state_month",         ["vehicle_type", "fuel_group"])
save(tables["reg_state_category_year"], "reg_state_category_year", ["vehicle_type"])
save(tables["reg_state_norm_year"],     "reg_state_norm_year",     ["vehicle_type"])
save(tables["reg_maker_state_year"],    "reg_maker_state_year",    ["vehicle_type"])
save(road_state,                        "road_safety_state_year")
save(road_nat,                          "road_safety_national_year")
save(checks,                            "_file_manifest")            # one row per raw file + check results

saved bronze/reg_state_month
saved bronze/reg_state_category_year
saved bronze/reg_state_norm_year
saved bronze/reg_maker_state_year
saved bronze/road_safety_state_year
saved bronze/road_safety_national_year
saved bronze/_file_manifest


## Step 8 · Read the Bronze tables back and confirm

In [8]:
for name in ["reg_state_month", "reg_state_category_year", "reg_state_norm_year",
             "reg_maker_state_year", "road_safety_state_year", "road_safety_national_year", "_file_manifest"]:
    df = spark.read.parquet((WH / "bronze" / name).as_uri())
    print(f"{name:28} {df.count():>9,} rows   columns: {', '.join(df.columns)[:110]}")

reg_state_month                 44,548 rows   columns: row_label, value_raw, column_label, is_total, dataset, source_file, load_ts, vehicle_type, fuel_group
reg_state_category_year          9,324 rows   columns: row_label, value_raw, column_label, is_total, dataset, fuel_group, source_file, load_ts, vehicle_type
reg_state_norm_year             13,986 rows   columns: row_label, value_raw, column_label, is_total, dataset, fuel_group, source_file, load_ts, vehicle_type
reg_maker_state_year           547,119 rows   columns: row_label, value_raw, column_label, is_total, dataset, fuel_group, source_file, load_ts, vehicle_type
road_safety_state_year             400 rows   columns: sl_no, state_raw, year_label, value_raw, year, measure, is_total, source_file, load_ts
road_safety_national_year           13 rows   columns: id, year, road_accidents_000, road_accident_deaths_000, road_accident_injuries_000, registered_vehicles_000, r
_file_manifest                      84 rows   columns: source_fi

In [9]:
# A peek: Kerala, EV, Non-Transport, 2025 (still raw text values - Silver converts them)
bm = spark.read.parquet((WH / "bronze" / "reg_state_month").as_uri())
(bm.filter((F.col("row_label") == "Kerala") & (F.col("fuel_group") == "EV") &
           (F.col("vehicle_type") == "NT") & F.col("column_label").startswith("2025"))
   .select("row_label", "column_label", "value_raw", "source_file")
   .show(12, truncate=False))

+---------+------------+---------+------------------------+
|row_label|column_label|value_raw|source_file             |
+---------+------------+---------+------------------------+
|Kerala   |2025-Jan    |7658     |fuel_EV_NT_2023_2025.csv|
|Kerala   |2025-Feb    |6251     |fuel_EV_NT_2023_2025.csv|
|Kerala   |2025-Mar    |9567     |fuel_EV_NT_2023_2025.csv|
|Kerala   |2025-Apr    |7219     |fuel_EV_NT_2023_2025.csv|
|Kerala   |2025-May    |8286     |fuel_EV_NT_2023_2025.csv|
|Kerala   |2025-Jun    |8047     |fuel_EV_NT_2023_2025.csv|
|Kerala   |2025-Jul    |8280     |fuel_EV_NT_2023_2025.csv|
|Kerala   |2025-Aug    |10248    |fuel_EV_NT_2023_2025.csv|
|Kerala   |2025-Sep    |9289     |fuel_EV_NT_2023_2025.csv|
|Kerala   |2025-Oct    |8650     |fuel_EV_NT_2023_2025.csv|
|Kerala   |2025-Nov    |8163     |fuel_EV_NT_2023_2025.csv|
|Kerala   |2025-Dec    |8338     |fuel_EV_NT_2023_2025.csv|
+---------+------------+---------+------------------------+



## Done ✅

Your Bronze warehouse is in `IAIP\warehouse\bronze\`. Next notebook: **02 · Silver**: convert text to numbers, one agreed name for every state, fuel groups → fuel types, remove totals, and run the data-quality checks (E3).

In [10]:
spark.stop()